# CMS Hospital Readmission Risk Analysis
## Notebook 2 — Data Cleansing

**Input:**  `data/raw/cms_patients_raw.csv`  
**Output:** `data/clean/cms_patients_clean.csv` + `reports/cleansing_report.json`

---

### What this notebook does
Applies 15 data quality rules to the raw dataset, documents every issue found, corrects or flags each problem, and produces a certified clean dataset with a full audit trail.

| Severity | Rules | Meaning |
|----------|-------|---------|
| CRITICAL | R01 | Data cannot be used without fix |
| HIGH | R02–R06 | Significant quality issue, must correct |
| MEDIUM | R07–R09 | Moderate issue, correct or flag |
| LOW | R10–R11 | Minor issue, flag and document |
| INFO | R12–R15 | Standardisation and audit columns |

In [23]:
import os

# Set working directory
os.chdir(r"C:\Users\abelo\Desktop\cms_readmission")

# Create all folders
os.makedirs('data/raw',        exist_ok=True)
os.makedirs('data/clean',      exist_ok=True)
os.makedirs('data/engineered', exist_ok=True)
os.makedirs('data/outputs',    exist_ok=True)
os.makedirs('reports',         exist_ok=True)

print('Working directory:', os.getcwd())
print('Folders ready.')

Working directory: C:\Users\abelo\Desktop\cms_readmission
Folders ready.


### 2.1 — Import libraries

In [24]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

Libraries loaded successfully


### 2.2 — Load raw data

In [18]:
df = pd.read_csv(r"C:\Users\abelo\Desktop\cms_readmission\raw\cms_patients_raw.csv")
df['admit_date']     = pd.to_datetime(df['admit_date'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nNull values in raw data:')
nulls = df.isnull().sum()
print(nulls[nulls > 0])
print(f'\nBasic statistics:')
df[['age','length_of_stay_days','bmi','total_cost_usd']].describe().round(2)

Loaded: 5,000 rows × 20 columns

Null values in raw data:
length_of_stay_days    175
bmi                     25
dtype: int64

Basic statistics:


,age,length_of_stay_days,bmi,total_cost_usd
count,5000.00,4825.00,4975.00,5000.00
mean,67.13,6.33,28.53,29436.87
std,14.44,3.30,6.51,14703.40
min,-1.00,1.00,15.00,0.00
25%,58.00,3.90,24.00,17814.50
50%,68.00,5.70,28.50,29803.00
75%,77.00,8.20,33.00,39125.25
max,95.00,26.10,51.90,109210.00


### 2.3 — Define logging function

In [19]:
issues_log   = []
rows_flagged = set()

def log_rule(rule_id, description, severity, n_affected, action):
    issues_log.append({
        'rule_id': rule_id, 'description': description,
        'severity': severity, 'records_affected': int(n_affected),
        'action': action,
    })
    label = {'CRITICAL':'CRIT','HIGH':'HIGH','MEDIUM':'MED ','LOW':'LOW ','INFO':'INFO'}.get(severity, severity)
    print(f'  [{label}] {rule_id}: {description}')
    print(f'         → {n_affected} record(s) → {action}')

### 2.4 — Apply all 15 data quality rules

In [20]:
print('Running data quality rules...\n')

# R01 — Duplicate patient IDs
dupes = df.duplicated(subset='patient_id', keep='first')
log_rule('R01','Duplicate patient_id records','CRITICAL', dupes.sum(),
         'Drop duplicates, keep first occurrence')
if dupes.sum(): rows_flagged.update(df[dupes].index); df = df[~dupes].copy()

# R02 — Invalid age
mask = (df['age'] < 18) | (df['age'] > 95)
log_rule('R02','Age outside valid range [18–95]','HIGH', mask.sum(),
         'Impute with diagnosis-group median')
rows_flagged.update(df[mask].index)
df.loc[mask, 'age'] = np.nan
df['age'] = df.groupby('primary_diagnosis')['age'].transform(
    lambda x: x.fillna(x.median())).round(0).astype(int)

# R03 — LOS out of range
mask = (df['length_of_stay_days'] < 1) | (df['length_of_stay_days'] > 30)
log_rule('R03','LOS outside valid range [1–30 days]','HIGH', mask.sum(),
         'Cap at bounds; flag with los_capped_flag')
rows_flagged.update(df[mask].index)
df['los_capped_flag'] = mask.astype(int)
df['length_of_stay_days'] = df['length_of_stay_days'].clip(1, 30)

# R04 — LOS vs date difference mismatch
df['_computed_los'] = (df['discharge_date'] - df['admit_date']).dt.days.round(1)
mismatch = abs(df['_computed_los'] - df['length_of_stay_days']) > 1
log_rule('R04','LOS field vs date diff disagree by >1 day','MEDIUM', mismatch.sum(),
         'Recompute LOS from admit/discharge dates')
rows_flagged.update(df[mismatch].index)
df.loc[mismatch, 'length_of_stay_days'] = df.loc[mismatch, '_computed_los']
df.drop(columns=['_computed_los'], inplace=True)

# R05 — Invalid BMI
mask = (df['bmi'] < 10) | (df['bmi'] > 60)
log_rule('R05','BMI outside plausible range [10–60]','HIGH', mask.sum(),
         'Impute with diagnosis-group median')
rows_flagged.update(df[mask].index)
df.loc[mask, 'bmi'] = np.nan
df['bmi'] = df.groupby('primary_diagnosis')['bmi'].transform(
    lambda x: x.fillna(x.median())).round(1)
df['bmi'] = df['bmi'].fillna(df['bmi'].median()).round(1)

# R06 — Zero or negative cost
mask = df['total_cost_usd'] <= 0
log_rule('R06','Total cost <= $0 (invalid)','HIGH', mask.sum(),
         'Impute with diagnosis-group median cost')
rows_flagged.update(df[mask].index)
df.loc[mask, 'total_cost_usd'] = np.nan
df['total_cost_usd'] = df.groupby('primary_diagnosis')['total_cost_usd'].transform(
    lambda x: x.fillna(x.median())).round(0).astype(int)

# R07 — Cost extreme outlier
Q1, Q3 = df['total_cost_usd'].quantile(0.25), df['total_cost_usd'].quantile(0.75)
upper   = Q3 + 3 * (Q3 - Q1)
mask    = df['total_cost_usd'] > upper
log_rule('R07',f'Total cost extreme outlier (> ${upper:,.0f}, 3×IQR)','MEDIUM', mask.sum(),
         'Flag with cost_outlier_flag; retain value')
rows_flagged.update(df[mask].index)
df['cost_outlier_flag'] = mask.astype(int)

# R08 — LOS extreme outlier
mask = df['length_of_stay_days'] > 20
log_rule('R08','LOS extreme outlier (> 20 days)','LOW', mask.sum(),
         'Flag with los_outlier_flag; clinically plausible, retain')
rows_flagged.update(df[mask].index)
df['los_outlier_flag'] = mask.astype(int)

# R09 — Null check
remaining_nulls = df.isnull().sum().sum()
log_rule('R09','Remaining nulls after imputation','INFO', remaining_nulls,
         'All nulls resolved' if remaining_nulls == 0 else 'Manual review required')

# R10 — Categorical whitespace and case
cat_cols = ['gender','ethnicity','hospital','primary_diagnosis','payer','discharge_disposition']
ws_total = 0
for col in cat_cols:
    before = df[col].copy()
    df[col] = df[col].str.strip().str.title()
    ws_total += (before != df[col]).sum()
log_rule('R10','Whitespace / case inconsistencies in categorical fields','LOW', ws_total,
         'Strip whitespace, apply title case')

# R11 — Date format standardisation
df['admit_date']     = df['admit_date'].dt.strftime('%Y-%m-%d')
df['discharge_date'] = df['discharge_date'].dt.strftime('%Y-%m-%d')
log_rule('R11','Standardise date format to ISO 8601','INFO', len(df),
         'Applied to admit_date and discharge_date')

# R12 — Re-derive temporal features
adt = pd.to_datetime(df['admit_date'])
df['admit_year']    = adt.dt.year
df['admit_month']   = adt.dt.month
df['admit_quarter'] = adt.dt.quarter
df['admit_dow']     = adt.dt.day_name()
df['admit_week']    = adt.dt.isocalendar().week.astype(int)
log_rule('R12','Re-derive temporal features from corrected dates','INFO', len(df),
         'admit_year/month/quarter/dow/week refreshed')

# R13 — Re-derive binary clinical flags
df['is_elderly']           = (df['age'] >= 75).astype(int)
df['is_high_comorbidity']  = (df['comorbidity_index'] >= 5).astype(int)
df['is_polypharmacy']      = (df['num_medications'] >= 10).astype(int)
df['is_frequent_flyer']    = (df['prior_admissions_12mo'] >= 3).astype(int)
df['is_long_stay']         = (df['length_of_stay_days'] >= 8).astype(int)
df['composite_risk_flags'] = (
    df['is_elderly'] + df['is_high_comorbidity'] + df['is_polypharmacy']
    + df['is_frequent_flyer'] + df['is_long_stay']
)
log_rule('R13','Re-derive binary clinical flags from cleaned values','INFO', len(df),
         'is_elderly, is_high_comorbidity, is_polypharmacy, is_frequent_flyer, is_long_stay')

# R14 — DQ audit columns
df['dq_flag_count']    = df[['los_capped_flag','los_outlier_flag','cost_outlier_flag']].sum(axis=1)
df['dq_record_status'] = df['dq_flag_count'].apply(lambda x: 'Clean' if x == 0 else 'Flagged')
log_rule('R14','Add DQ audit columns','INFO', len(df),
         'dq_flag_count and dq_record_status added')

Running data quality rules...

  [CRIT] R01: Duplicate patient_id records
         → 0 record(s) → Drop duplicates, keep first occurrence
  [HIGH] R02: Age outside valid range [18–95]
         → 25 record(s) → Impute with diagnosis-group median
  [HIGH] R03: LOS outside valid range [1–30 days]
         → 0 record(s) → Cap at bounds; flag with los_capped_flag
  [MED ] R04: LOS field vs date diff disagree by >1 day
         → 0 record(s) → Recompute LOS from admit/discharge dates
  [HIGH] R05: BMI outside plausible range [10–60]
         → 0 record(s) → Impute with diagnosis-group median
  [HIGH] R06: Total cost <= $0 (invalid)
         → 25 record(s) → Impute with diagnosis-group median cost
  [MED ] R07: Total cost extreme outlier (> $102,181, 3×IQR)
         → 1 record(s) → Flag with cost_outlier_flag; retain value
  [LOW ] R08: LOS extreme outlier (> 20 days)
         → 7 record(s) → Flag with los_outlier_flag; clinically plausible, retain
  [INFO] R09: Remaining nulls after imputati

### 2.5 — Cleansing summary

In [21]:
total_touched = len(rows_flagged)
clean_count   = (df['dq_record_status'] == 'Clean').sum()
flagged_count = (df['dq_record_status'] == 'Flagged').sum()

print('=' * 55)
print('  CLEANSING SUMMARY')
print('=' * 55)
print(f'  Rules applied:             14')
print(f'  Records touched:           {total_touched:,} ({total_touched/len(df)*100:.1f}%)')
print(f'  Clean records (0 flags):   {clean_count:,} ({clean_count/len(df)*100:.1f}%)')
print(f'  Flagged records (1+):      {flagged_count:,} ({flagged_count/len(df)*100:.1f}%)')
print(f'  Remaining nulls:           {df.isnull().sum().sum()}')
print(f'  Final shape:               {df.shape[0]:,} rows × {df.shape[1]} columns')
print('=' * 55)

  CLEANSING SUMMARY
  Rules applied:             14
  Records touched:           57 (1.1%)
  Clean records (0 flags):   4,993 (99.9%)
  Flagged records (1+):      7 (0.1%)
  Remaining nulls:           175
  Final shape:               5,000 rows × 36 columns


### 2.6 — Save clean dataset and audit report

In [25]:
df.to_csv('data/clean/cms_patients_clean.csv', index=False)

report = {
    'run_timestamp':   datetime.now().isoformat(),
    'source_file':     'data/raw/cms_patients_raw.csv',
    'output_file':     'data/clean/cms_patients_clean.csv',
    'source_rows':     5000,
    'output_rows':     int(len(df)),
    'rules_applied':   14,
    'records_touched': int(total_touched),
    'clean_records':   int(clean_count),
    'flagged_records': int(flagged_count),
    'remaining_nulls': int(df.isnull().sum().sum()),
    'output_columns':  int(df.shape[1]),
    'rules':           issues_log,
}
os.makedirs('reports', exist_ok=True)
with open('reports/cleansing_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Files saved:')
print(f'  data/clean/cms_patients_clean.csv  — {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'  reports/cleansing_report.json')
print(f'\nPreview of clean dataset:')
df[['patient_id','age','primary_diagnosis','length_of_stay_days','total_cost_usd',
    'readmitted_30d','dq_record_status']].head()

Files saved:
  data/clean/cms_patients_clean.csv  — 5,000 rows × 36 columns
  reports/cleansing_report.json

Preview of clean dataset:


,patient_id,age,primary_diagnosis,length_of_stay_days,total_cost_usd,readmitted_30d,dq_record_status
0,PT000001,74,Cabg,4.4,11937,0,Clean
1,PT000002,66,Hip/Knee Replacement,4.4,10524,0,Clean
2,PT000003,77,Cabg,8.4,44368,1,Clean
3,PT000004,89,Pneumonia,2.5,6182,0,Clean
4,PT000005,64,Heart Failure,3.7,28081,1,Clean


---
**Notebook 2 complete.** Output saved to `data/clean/cms_patients_clean.csv`

Next → **Notebook 3: Feature Engineering**